In [2]:
import sys
print(sys.executable)
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings
warnings.filterwarnings("ignore")

# ── 0. DATEI LADEN ───────────────────────────────────────────
# Pfad anpassen!
CSV_PATH = "../en.openfoodfacts.org.products 2.csv"

cols = ["manufacturing_places", "manufacturing_places_tags"]
df = pd.read_csv(CSV_PATH, sep="	", usecols=cols, low_memory=False)

# Füllrate aller Spalten
print(df[cols].notna().mean().round(2) * 100)

# Alle Paare auf Überlappung prüfen
import itertools

for col1, col2 in itertools.combinations(cols, 2):
    beide = df[[col1, col2]].notna().all(axis=1).sum()
    print(f"{col1} + {col2}: beide befüllt in {beide} Zeilen")

# Wie oft sind manufacturing_places UND manufacturing_places_tags gleichzeitig befüllt?
pd.crosstab(
    df['manufacturing_places'].notna(),
    df['manufacturing_places_tags'].notna(),
    margins=True
)

/opt/homebrew/opt/python@3.11/bin/python3.11
manufacturing_places         5.0
manufacturing_places_tags    5.0
dtype: float64
manufacturing_places + manufacturing_places_tags: beide befüllt in 219346 Zeilen


manufacturing_places_tags,False,True,All
manufacturing_places,,,
False,4281944,3,4281947
True,64,219346,219410
All,4282008,219349,4501357


In [3]:
# Zeilen wo mehrere Spalten gleichzeitig befüllt sind
mask = df[cols].notna().sum(axis=1) >= 2
df[cols][mask].head(20)

,manufacturing_places,manufacturing_places_tags
15,France,france
24,"Saint-Yrieix,France","saint-yrieix,france"
32,Canada,canada
42,Argentine,argentine
45,uk,uk
54,"87500,France","87500,france"
89,USA,usa
90,Ancenis,ancenis
98,"Fort Worth, TX","fort-worth,tx"
103,Canada,canada


In [4]:
df['manufacturing_places_combined'] = (
    df['manufacturing_places_tags']
    .fillna(df['manufacturing_places'])
)

In [5]:
df.to_parquet('manufacturing_places_bereinigung.parquet', index=False)